# B1 - MCTS from scratch: how search turns a raw net into a planner

AlphaZero's search is **PUCT MCTS**. Each *simulation* walks down the tree, at each
decision node picking the move that maximises

$$\text{score}(a) = Q(a) + c_\text{puct}\, P(a)\, \frac{\sqrt{N_\text{parent}}}{1 + N(a)}$$

- $Q(a)$ exploits (how good has this move looked?),
- $P(a)$ is the network's prior and $\sqrt{N}/(1+N(a))$ explores (try moves we
  haven't visited much).

When a fresh leaf is reached, the **network** evaluates it (prior + value); the
value is backed up the path (sign-flipped for the opponent). The **visit counts**
at the root become a "search-refined" policy - a bit smarter than the raw prior.

Below we watch this happen with an *untrained* network, so any good behaviour comes
purely from search.

In [1]:
import numpy as np, torch
from azl.games.tictactoe import TicTacToe
from azl.network import net_for_game
from azl.mcts import MCTS, select_action

torch.manual_seed(0)
net = net_for_game(TicTacToe)          # random weights!

# Position where X (player 0) can win immediately by playing cell 2 (top row).
s = TicTacToe(board=(1, 1, 0, 2, 2, 0, 0, 0, 0), player=0)
print(s.render(), "\n")

prior, value = net.predict(s.encode(), s.legal_actions())
print("raw network prior:", {a: round(p, 2) for a, p in prior.items()}, " value:", round(value, 2))

X X .
O O .
. . . 

raw network prior: {2: 0.2, 5: 0.19, 6: 0.21, 7: 0.19, 8: 0.22}  value: 0.08


In [2]:
mcts = MCTS(net, num_simulations=200, rng=np.random.default_rng(0))
visit_probs, root_value, visit_counts = mcts.search(s)
print("visit counts   :", visit_counts)
print("visit policy   :", {a: round(p, 2) for a, p in visit_probs.items()})
print("chosen move    :", select_action(visit_counts, temperature=0.0), "(2 is the win)")
print("root value     :", round(root_value, 2))

visit counts   : {2: 187, 8: 3, 6: 3, 5: 4, 7: 3}
visit policy   : {2: 0.94, 8: 0.01, 6: 0.01, 5: 0.02, 7: 0.01}
chosen move    : 2 (2 is the win)
root value     : 0.93


Even with random weights, **search finds the winning move 2**: simulations that
play 2 reach a terminal win (+1) and that value backs up, so the visit count piles
onto move 2. Compare the flat *prior* with the peaked *visit policy* - that gap is
the "search is smarter than the net" signal the policy head will learn to imitate
(B2).

### Things to tweak
- Lower `num_simulations` to 10 - search gets noisier and may miss the win.
- Raise `c_puct` - more exploration, visits spread out.

### Maps to raccoon
This is `azl/search/mcts.py` <-> `raccoon/search/mcts.py`. Same PUCT formula, same
sign-flipping backup. raccoon adds batched inference and virtual loss for speed; the
logic is identical.

In [3]:
import inspect
from azl import mcts
print(inspect.getsource(mcts.MCTS._puct_action))

    def _puct_action(self, node: Node) -> int:
        sqrt_n = math.sqrt(node.visit_count)
        best_score, best_action = -float("inf"), -1
        for action, prior in node.child_priors.items():
            child = node.children.get(action)
            if child is not None and child.visit_count > 0:
                # child's Q is from the child's perspective; flip if it's the opponent's.
                same = child.state.current_player() == node.state.current_player()
                q_parent = child.q_value if same else -child.q_value
                n = child.visit_count
            else:
                q_parent, n = 0.0, 0
            score = q_parent + self.c_puct * prior * sqrt_n / (1 + n)
            if score > best_score:
                best_score, best_action = score, action
        return best_action

